# Perhitungan Sentralitas

## Import Library

In [103]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

## Load Data

In [104]:
# pd.set_option('display.max_colwidth', None)

df = pd.read_csv('../data/processed/edges.csv', header=None)
df.columns = ['Guru', 'Murid', 'Weight', 'Inverse_Weight']
df = df.iloc[1:].reset_index(drop=True)

display(df)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_28988\3877324507.py:3: DtypeWarning: Columns (0: 2, 1: 3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/processed/edges.csv', header=None)


,Guru,Murid,Weight,Inverse_Weight
0,جابر بن زيد الازدي,ابو عبيدة مسلم بن ابو كريمة التميمي,1,1.0
1,عبد الله بن عباس,جابر بن زيد الازدي,1,1.0
2,جابر بن زيد,ابو عبيدة,522,0.0019157088122605363
3,عائشة,جابر بن زيد,68,0.014705882352941176
4,ابو هريرة,جابر بن زيد,76,0.013157894736842105
...,...,...,...,...
776810,ابي مسعود البدري,حذيفة بن اليمان,1,1.0
776811,سعيد بن العاص,ابي مسعود البدري,1,1.0
776812,ابو مسعود,سعيد بن العاص,1,1.0
776813,عبد الله,علي بن علقمة,1,1.0


In [105]:
print(df.dtypes)

Guru                 str
Murid                str
Weight            object
Inverse_Weight    object
dtype: object


In [106]:
df['Weight'] = pd.to_numeric(df['Weight'])
df['Inverse_Weight'] = pd.to_numeric(df['Inverse_Weight'])
print(df.dtypes)

Guru                  str
Murid                 str
Weight              int64
Inverse_Weight    float64
dtype: object


### Bentuk graph

In [107]:
G = nx.from_pandas_edgelist(
    df,
    source='Murid',
    target='Guru',
    edge_attr=['Weight', 'Inverse_Weight'],
    create_using=nx.DiGraph()
)

In [108]:
print("Jumlah node :", G.number_of_nodes())
print("Jumlah edge :", G.number_of_edges())

Jumlah node : 177206
Jumlah edge : 776815


In [109]:
display(df)

,Guru,Murid,Weight,Inverse_Weight
0,جابر بن زيد الازدي,ابو عبيدة مسلم بن ابو كريمة التميمي,1,1.000000
1,عبد الله بن عباس,جابر بن زيد الازدي,1,1.000000
2,جابر بن زيد,ابو عبيدة,522,0.001916
3,عائشة,جابر بن زيد,68,0.014706
4,ابو هريرة,جابر بن زيد,76,0.013158
...,...,...,...,...
776810,ابي مسعود البدري,حذيفة بن اليمان,1,1.000000
776811,سعيد بن العاص,ابي مسعود البدري,1,1.000000
776812,ابو مسعود,سعيد بن العاص,1,1.000000
776813,عبد الله,علي بن علقمة,1,1.000000


In [110]:
# Hapus self-loop
G.remove_edges_from(nx.selfloop_edges(G))

In [111]:
print("Jumlah node setelah hapus self-loop:", G.number_of_nodes())
print("Jumlah edge setelah hapus self-loop:", G.number_of_edges())

Jumlah node setelah hapus self-loop: 177206
Jumlah edge setelah hapus self-loop: 776815


## Menghitung Sentralitas

### Menghitung In Degree, Out Degree, Degree, dan Degree Centrality

In [112]:
# Degree (node atau total hubungan)
degree_dict = dict(G.degree())


# In-Degree (peran perawi sebagai Guru)
# In-Degree = jumlah murid
in_degree_dict = dict(G.in_degree())


# Out-Degree (peran perawi sebagai Murid)
# Out-Degree = jumlah guru
out_degree_dict = dict(G.out_degree())


# Degree Centrality (tingkat keterhubungan node dalam jaringan.)
degree_centrality_dict = nx.degree_centrality(G)

# Gabungkan ke DataFrame

centrality_df = pd.DataFrame({
    'Perawi': list(G.nodes()),

    'In_Degree_Jumlah_Murid': [
        in_degree_dict[node]
        for node in G.nodes()
    ],

    'Out_Degree_Jumlah_Guru': [
        out_degree_dict[node]
        for node in G.nodes()
    ],

    'Degree_Total_Hubungan': [
        degree_dict[node]
        for node in G.nodes()
    ],

    'Degree_Centrality': [
        degree_centrality_dict[node]
        for node in G.nodes()
    ]
})

# Urutkan berdasarkan Degree Centrality
centrality_df = centrality_df.sort_values(
    by='Degree_Centrality',
    ascending=False
)


# Reset index
centrality_df = centrality_df.reset_index(drop=True)


# Tampilkan hasil
display(centrality_df.head(20))

,Perawi,In_Degree_Jumlah_Murid,Out_Degree_Jumlah_Guru,Degree_Total_Hubungan,Degree_Centrality
0,ابو هريرة,3530,1291,4821,0.027206
1,سفيان,2294,2391,4685,0.026438
2,شعبة,2121,2198,4319,0.024373
3,ابن عباس,2662,946,3608,0.020361
4,عائشة,2334,733,3067,0.017308
5,الاعمش,1748,1306,3054,0.017234
6,ابن عمر,2246,796,3042,0.017167
7,ابو عبد الله الحافظ,786,2247,3033,0.017116
8,الزهري,1554,1466,3020,0.017042
9,عبد الله,1491,1364,2855,0.016111


### Menghitung Eigenvector Centrality (Kualitas koneksi)

In [136]:
# Hitung Eigenvector Centrality (tingkat pengaruh node dalam jaringan atau seberapa penting node tersebut dalam jaringan)
eigenvector_dict = nx.eigenvector_centrality(
    G,
    max_iter=1000,
    tol=1e-05,
    weight='Weight'
)

# Ubah menjadi DataFrame
eigenvector_df = pd.DataFrame({
    'Perawi': list(eigenvector_dict.keys()),
    'Eigenvector_Centrality': list(eigenvector_dict.values())
})


# Urutkan dari terbesar
eigenvector_df = eigenvector_df.sort_values(
    by='Eigenvector_Centrality',
    ascending=False
)

# Reset index
eigenvector_df = eigenvector_df.reset_index(drop=True)

# Tampilkan hasil
display(eigenvector_df.head(20))

,Perawi,Eigenvector_Centrality
0,ابن عباس,0.785159
1,مجاهد,0.458519
2,ابن عمر,0.326439
3,ابن ابو نجيح,0.175712
4,عبد الرحمن بن ابو ليلي,0.119758
5,رقاء,0.086012
6,عائشة,0.083841
7,كعب بن عجرة,0.044856
8,ادم,0.042366
9,علقمة,0.032596


In [114]:
# mengecek jumlah node setelah semua proses
print(G.number_of_nodes())

177206


In [115]:
# mengecek data  yang digunakan untuk menghitung eigenvector centrality
print(len(eigenvector_dict))

177206


### Menghitung data dengan 4000 data teratas untuk Closeness dan Between 

In [116]:
# # Ambil 40000 data teratas
# df_40000 = df.head(40000)

# # Buat graph dari kolom Guru dan Murid
# G_sample = nx.from_pandas_edgelist(
#     df_40000,
#     source='Guru',
#     target='Murid',
#     edge_attr=['Weight', 'Inverse_Weight'],
#     create_using=nx.DiGraph()
# )

# # Tampilkan jumlah node dan edge
# print("Jumlah Node :", G_sample.number_of_nodes())
# print("Jumlah Edge :", G_sample.number_of_edges())

In [117]:
# display(df_40000)

### Menghitung Closeness Centrality

In [ ]:
import networkx as nx
import pandas as pd
from tqdm import tqdm

# Ambil 3000 data teratas
df_sample = df.head(3000)

# Buat graph berarah Guru -> Murid
G_sample = nx.from_pandas_edgelist(
    df_sample,
    source='Guru',
    target='Murid',
    create_using=nx.DiGraph()
)

print("Jumlah Node :", G_sample.number_of_nodes())
print("Jumlah Edge :", G_sample.number_of_edges())

def closeness_for_node(node):
    return node, nx.closeness_centrality(G_sample, u=node, distance="Inverse_Weight")

nodes = list(G_sample.nodes())
# Hitung closeness centrality
closeness = {}

for node in tqdm(G_sample.nodes(), desc="Menghitung Closeness Centrality"):
    closeness[node] = nx.closeness_centrality(G_sample, u=node, distance="Inverse_Weight")

print("Closeness Centrality selesai.")

Jumlah Node : 1608
Jumlah Edge : 3000


Menghitung Closeness Centrality: 100%|██████████| 1608/1608 [00:24<00:00, 66.56it/s]

Closeness Centrality selesai.


In [151]:
closeness_df = pd.DataFrame(
    closeness.items(),
    columns=['Perawi', 'Closeness Centrality']
)

closeness_df = closeness_df.sort_values(
    by='Closeness Centrality',
    ascending=False
).reset_index(drop=True)

display(closeness_df.head(20))

,Perawi,Closeness Centrality
0,ابو حنيفة,0.261548
1,شعبة,0.220008
2,عطاء بن السائب,0.218804
3,النعمان بن ثابت,0.217206
4,سفيان,0.217165
5,زفر,0.216274
6,ابراهيم,0.216072
7,عبد الرحمن,0.214672
8,الشعبي,0.213094
9,عبد الله بن عمرو,0.212236


### Menghitung Between